<div style="background-color:#f2f2f2; border-left: 8px solid #2e8b57; padding: 20px; border-radius: 10px; font-family: Arial, sans-serif;">

<center>

<h1 style="color:#2e8b57;">Diseño de experimentos</h1>
<h2 style="color:#2e8b57;">Instituto Mexicano del Petróleo</h2>
<h3 style="color:#2e8b57;">Primavera 2026</h3>

<p style="font-size:16px; color:#333;">
<strong>Dra. Gabriela Berenice Díaz Cortés</strong><br>
<em>gbdiaz@imp.mx</em>
</p>

<p style="font-size:15px; color:#555;">
Dirección de Investigación <br>
Gerencia de Investigación en Explotación
</p>

</center>

</div>

# PCA y SVD para Machine Learning
## Fundamentos matemáticos, reconstrucción de datos y Eigenfaces
### Motivación

En numerosos problemas modernos, cada observación contiene cientos o miles de variables.

Ejemplos:

- imágenes digitales,
- señales de sensores,
- perfiles genómicos,
- datos experimentales multivariados.

Cuando el número de variables es grande, aparecen problemas de:

- **redundancia:** variables altamente correlacionadas contienen información repetida,
- **ruido:** variables con poca variación aportan más interferencia que señal,
- **costo computacional:** modelos con muchas variables son lentos e inestables,
- **dificultad de visualización:** no es posible graficar directamente en más de tres dimensiones.

La pregunta central es:

> ¿Existe una representación más compacta que conserve la mayor parte de la información?

La respuesta conduce al **Análisis de Componentes Principales (PCA)** y a la **Descomposición en Valores Singulares (SVD)**.


In [ ]:
# Ejemplo motivacional: por que necesitamos PCA
#
# Vamos a generar DOS escenarios contrastantes:
#
#   Escenario A: 4 variables CORRELACIONADAS (provienen de una sola fuente)
#   Escenario B: 4 variables INDEPENDIENTES  (cada una aporta informacion distinta)
#
# La pregunta es: si en el Escenario A toda la informacion viene de una sola
# fuente, para que guardar 4 columnas? PCA permite descubrir esto
# y representar los datos con mucho menos.

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
m = 100
t = np.linspace(0, 10, m)  # fuente subyacente comun

# --- Escenario A: variables correlacionadas ---
# Todas son versiones ruidosas de la misma señal t
A1 = 1.0 * t + np.random.normal(0, 0.3, m)
A2 = 2.1 * t + np.random.normal(0, 0.5, m)
A3 = 0.8 * t + np.random.normal(0, 0.2, m)
A4 = 3.0 * t + np.random.normal(0, 0.7, m)

# --- Escenario B: variables independientes ---
# Cada variable tiene su propia fuente aleatoria sin relacion con las demas
B1 = np.random.normal(0, 3, m)
B2 = np.random.normal(0, 3, m)
B3 = np.random.normal(0, 3, m)
B4 = np.random.normal(0, 3, m)

# ----------------------------------------------------------------
# Figura 1: las 4 series en el tiempo
# ----------------------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(14, 5))

nombres = ['x1', 'x2', 'x3', 'x4']
datos_A = [A1, A2, A3, A4]
datos_B = [B1, B2, B3, B4]

for j in range(4):
    axes[0, j].plot(t, datos_A[j], color='steelblue', lw=1.2)
    axes[0, j].set_title('Escenario A: ' + nombres[j], fontsize=9)
    axes[0, j].set_xlabel('observacion')
    axes[0, j].set_ylabel('valor')

    axes[1, j].plot(t, datos_B[j], color='tomato', lw=1.2)
    axes[1, j].set_title('Escenario B: ' + nombres[j], fontsize=9)
    axes[1, j].set_xlabel('observacion')
    axes[1, j].set_ylabel('valor')

axes[0, 0].set_ylabel('Escenario A\n(correlacionadas)', fontsize=9)
axes[1, 0].set_ylabel('Escenario B\n(independientes)', fontsize=9)
plt.suptitle('Las 4 variables a lo largo de las observaciones', fontsize=11)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------
# Figura 2: diagrama de dispersion x1 vs x2 en cada escenario
# ----------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.scatter(A1, A2, alpha=0.5, s=20, color='steelblue')
ax1.set_xlabel('x1')
ax1.set_ylabel('x2')
ax1.set_title('Escenario A (correlacionadas)\nx1 vs x2: relacion LINEAL')
ax1.text(0.05, 0.90,
         'Correlacion: ' + str(round(float(np.corrcoef(A1, A2)[0, 1]), 3)),
         transform=ax1.transAxes, fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

ax2.scatter(B1, B2, alpha=0.5, s=20, color='tomato')
ax2.set_xlabel('x1')
ax2.set_ylabel('x2')
ax2.set_title('Escenario B (independientes)\nx1 vs x2: NUBE sin estructura')
ax2.text(0.05, 0.90,
         'Correlacion: ' + str(round(float(np.corrcoef(B1, B2)[0, 1]), 3)),
         transform=ax2.transAxes, fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.suptitle('Dispersion entre variables: correlacion vs independencia', fontsize=11)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------
# Figura 3: matriz de correlacion como mapa de calor
# ----------------------------------------------------------------
XA = np.column_stack([A1, A2, A3, A4])
XB = np.column_stack([B1, B2, B3, B4])

corrA = np.corrcoef(XA.T)   # corrcoef espera variables en filas
corrB = np.corrcoef(XB.T)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

im1 = ax1.imshow(corrA, cmap='RdYlGn', vmin=-1, vmax=1)
ax1.set_xticks(range(4)); ax1.set_yticks(range(4))
ax1.set_xticklabels(nombres); ax1.set_yticklabels(nombres)
ax1.set_title('Escenario A: correlaciones altas\n-> informacion REDUNDANTE')
for i in range(4):
    for j in range(4):
        ax1.text(j, i, str(round(corrA[i, j], 2)),
                 ha='center', va='center', fontsize=10,
                 color='black')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(corrB, cmap='RdYlGn', vmin=-1, vmax=1)
ax2.set_xticks(range(4)); ax2.set_yticks(range(4))
ax2.set_xticklabels(nombres); ax2.set_yticklabels(nombres)
ax2.set_title('Escenario B: correlaciones ~0\n-> informacion COMPLEMENTARIA')
for i in range(4):
    for j in range(4):
        ax2.text(j, i, str(round(corrB[i, j], 2)),
                 ha='center', va='center', fontsize=10,
                 color='black')
plt.colorbar(im2, ax=ax2)

plt.suptitle('Matriz de correlacion entre variables', fontsize=11)
plt.tight_layout()
plt.show()

print('Escenario A: correlaciones muy altas (cercanas a 1.0)')
print('  -> Las 4 variables dicen casi lo mismo.')
print('  -> Con 1 sola dimension podriamos capturar casi toda la informacion.')
print()
print('Escenario B: correlaciones cercanas a 0')
print('  -> Cada variable aporta informacion distinta.')
print('  -> Necesitamos las 4 dimensiones para no perder informacion.')
print()
print('PCA ataca el Escenario A: descubre cuantas dimensiones "reales" hay')
print('y descarta las redundantes.')


# Matriz de datos

Sea:

$$
X \in \mathbb{R}^{m \times n}
$$

representada por:

$$
X =
\begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1n} \\
x_{21} & x_{22} & \cdots & x_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
x_{m1} & x_{m2} & \cdots & x_{mn}
\end{bmatrix}
$$

donde:

- $m$ es el número de observaciones (filas),
- $n$ es el número de variables medidas en cada observación (columnas).

La fila $i$ representa la observación $i$: un vector que contiene los valores de todas las variables para ese individuo.

La columna $j$ contiene los valores de la variable $j$ a lo largo de todas las observaciones.


In [ ]:
# Construccion explicita de la matriz de datos X
#
# Usamos el mismo ejemplo motivacional.
# Cada columna es una variable; cada fila es una observacion.

import numpy as np

np.random.seed(42)
m = 15   # observaciones (filas)
n = 4    # variables (columnas)
t = np.linspace(1, 10, m)

# Construimos X columna por columna y luego las apilamos
col1 = 1.0 * t + np.random.normal(0, 0.3, m)
col2 = 2.1 * t + np.random.normal(0, 0.5, m)
col3 = 0.5 * t + np.random.normal(0, 0.2, m)
col4 = 3.0 * t + np.random.normal(0, 0.7, m)

# numpy.column_stack apila vectores como columnas
X = np.column_stack([col1, col2, col3, col4])

print('Dimensiones de X:', X.shape)
print('  Filas    (observaciones) m =', X.shape[0])
print('  Columnas (variables)     n =', X.shape[1])
print()
print('Primeras 5 filas de X:')
print(np.round(X[:5, :], 3))
print()
print('La fila 0 (primera observacion):')
print('  x1 =', round(X[0, 0], 3),
      '  x2 =', round(X[0, 1], 3),
      '  x3 =', round(X[0, 2], 3),
      '  x4 =', round(X[0, 3], 3))


#  Centrado de datos

El objetivo de PCA es estudiar la **variabilidad** de los datos respecto a su promedio.

Sea la media por columnas:

$$
\mu_j = \frac{1}{m}\sum_{i=1}^{m} x_{ij}
\qquad j=1,2,\dots,n
$$

La matriz centrada se obtiene restando la media de cada columna:

$$
X_c = X - \mathbf{1}\mu^T
$$

donde $\mathbf{1}\in\mathbb{R}^{m}$ es un vector columna de unos.

Después del centrado, **cada columna de $X_c$ tiene media exactamente cero**.

El centrado garantiza que el origen del nuevo sistema de coordenadas coincida con el promedio de los datos, condición necesaria para que la varianza sea el criterio correcto de análisis.


In [ ]:
# Centrado de datos: calculo paso a paso
#
# Paso 1: calcular la media de cada columna
# Paso 2: restarla de cada fila

# --- Paso 1: media por columna ---
# numpy.mean con axis=0 calcula la media de cada columna
mu = np.mean(X, axis=0)

print('Media de cada variable:')
for j in range(n):
    print('  mu[' + str(j) + '] (variable x' + str(j+1) + ') = ' + str(round(mu[j], 4)))

# --- Paso 2: restar la media ---
# Cuando X tiene forma (m, n) y mu tiene forma (n,),
# numpy aplica la resta columna por columna automaticamente (broadcasting).
Xc = X - mu

print()
print('Media de cada columna en Xc (deben ser ~0):')
for j in range(n):
    print('  columna ' + str(j) + ': media = ' + str(round(np.mean(Xc[:, j]), 10)))

print()
print('Primeras 5 filas de Xc (datos centrados):')
print(np.round(Xc[:5, :], 4))


# Matriz de covarianza

Una vez centrados los datos, la dependencia lineal entre variables se mide mediante la **matriz de covarianza muestral**:

$$
S = \frac{1}{m-1} X_c^T X_c
$$

donde $S \in \mathbb{R}^{n \times n}$.

El elemento $(j, k)$ de $S$ es:

$$
s_{jk} = \operatorname{Cov}(x_j, x_k)
$$

Propiedades:

- La diagonal contiene las varianzas: $s_{jj} = \operatorname{Var}(x_j)$.
- $S$ es **simétrica**: $S = S^T$.
- $S$ es **semidefinida positiva**: todos sus eigenvalores son $\geq 0$.

**Relevancia para PCA:** Si varias variables están altamente correlacionadas, contienen información repetida. PCA busca nuevas variables no correlacionadas que concentren esa información en menos dimensiones.


In [ ]:
# Calculo manual de la matriz de covarianza
#
# S = (1 / (m-1)) * Xc^T * Xc
# Usamos numpy.dot para el producto matricial

m = Xc.shape[0]  # numero de observaciones
n = Xc.shape[1]  # numero de variables

# Paso 1: calcular Xc^T (transpuesta de Xc)
XcT = Xc.T               # forma: (n, m) = (4, 15)

# Paso 2: producto matricial XcT * Xc
# numpy.dot(A, B) calcula el producto matricial A * B
XcT_Xc = np.dot(XcT, Xc)  # forma: (n, n) = (4, 4)

# Paso 3: dividir por (m - 1) para estimacion muestral
S = XcT_Xc / (m - 1)

print('Matriz de covarianza S (4 x 4):')
print(np.round(S, 4))
print()
print('Verificacion de simetria (S - S^T debe ser la matriz cero):')
print(np.round(S - S.T, 10))
print()
print('Diagonal de S (varianzas de cada variable):')
for j in range(n):
    print('  Var(x' + str(j+1) + ') = ' + str(round(S[j, j], 4)))
print()
print('La covarianza entre x1 y x2 es:', round(S[0, 1], 4))
print('  -> valor positivo y alto: ambas variables crecen juntas.')


#  Eigenvalores y eigenvectores

Sea una matriz cuadrada $S \in \mathbb{R}^{n \times n}$.

Buscamos vectores no nulos $v \neq 0$ tales que al aplicar la transformación definida por $S$, la dirección del vector no cambie:

$$
Sv = \lambda v
$$

donde:

- $v$ se denomina **eigenvector** (vector propio),
- $\lambda$ se denomina **eigenvalor** (valor propio) asociado.

Para que exista una solución no trivial, la matriz $S - \lambda I$ debe ser singular:

$$
\det(S - \lambda I) = 0
$$

Esta es la **ecuación característica** de $S$.

En PCA, $S$ es la matriz de covarianza, que es simétrica y semidefinida positiva. Esto garantiza que:

- todos los eigenvalores son reales y $\lambda_i \geq 0$,
- los eigenvectores son ortogonales entre sí,
- el eigenvalor $\lambda_i$ mide la **varianza** de los datos en la dirección del eigenvector $v_i$.

Por ello se ordenan de mayor a menor:

$$
\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_n \geq 0
$$


In [ ]:
# Calculo de eigenvalores y eigenvectores de la matriz de covarianza
#
# numpy.linalg.eig(S) devuelve:
#   vals: arreglo con los eigenvalores
#   vecs: matriz cuyas COLUMNAS son los eigenvectores
#
# Los eigenvalores no vienen ordenados; hay que ordenarlos manualmente.

from numpy.linalg import eig

vals_raw, vecs_raw = eig(S)

# Los eigenvalores de una matriz de covarianza son reales,
# pero numpy puede devolver partes imaginarias numericamente pequeñas.
vals_raw = vals_raw.real
vecs_raw = vecs_raw.real

# Ordenar de mayor a menor
idx = np.argsort(vals_raw)[::-1]   # indices que ordenan de mayor a menor
vals = vals_raw[idx]               # eigenvalores ordenados
vecs = vecs_raw[:, idx]            # eigenvectores en las columnas, en el mismo orden

print('Eigenvalores (de mayor a menor):')
for i, lam in enumerate(vals):
    print('  lambda_' + str(i+1) + ' = ' + str(round(lam, 6)))

print()
print('Varianza explicada por cada componente:')
total = np.sum(vals)
for i, lam in enumerate(vals):
    print('  PC' + str(i+1) + ': ' + str(round(100 * lam / total, 2)) + ' %')

print()
print('Interpretacion:')
print('  La primera componente concentra casi toda la varianza.')
print('  Esto confirma que los datos tienen una sola dimension subyacente.')
print()

# Verificacion: S * v_1 debe ser igual a lambda_1 * v_1
v1 = vecs[:, 0]         # primer eigenvector (columna 0)
lam1 = vals[0]          # primer eigenvalor

Sv1   = np.dot(S, v1)   # S * v_1
lv1   = lam1 * v1       # lambda_1 * v_1

print('Verificacion S*v1 = lambda_1*v1:')
print('  S*v1   =', np.round(Sv1, 6))
print('  lam1*v1=', np.round(lv1, 6))
print('  Diferencia maxima:', round(np.max(np.abs(Sv1 - lv1)), 10))


In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, ax = plt.subplots(figsize=(9,5))

x = np.arange(1, len(vals)+1)

# Curva principal
ax.plot(x, vals, marker='o', linewidth=2)

ax.set_xlabel('Componente principal')
ax.set_ylabel('Eigenvalor')
ax.set_title('Eigenvalores ordenados de mayor a menor')
ax.set_xticks(x)
ax.grid(True, alpha=0.3)

# -------------------------
# Recuadro logaritmico
# -------------------------
axins = inset_axes(ax, width="38%", height="38%", loc='upper right')

axins.plot(x, vals, marker='o', linewidth=1.5)
axins.set_yscale('log')
axins.set_xticks([])
axins.grid(True, alpha=0.25)

axins.set_title('Escala log', fontsize=9)

plt.show()

# Componentes principales

Las **componentes principales** son las direcciones de máxima variabilidad de los datos.

Cada componente principal corresponde a un eigenvector de la matriz de covarianza, ordenado por eigenvalor decreciente.

La varianza explicada por la $i$-ésima componente se cuantifica como fracción de la varianza total:

$$
p_i = \frac{\lambda_i}{\sum_{j=1}^{n} \lambda_j}
$$

La **varianza acumulada** hasta la $k$-ésima componente es:

$$
P_k = \sum_{i=1}^{k} p_i
$$

En la práctica, se selecciona el menor $k$ tal que $P_k \geq$ un umbral deseado (por ejemplo, $95\%$).

Esta elección implica un equilibrio entre compresión (pocas componentes) y fidelidad (poca pérdida de información).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

varianza_individual = vals / np.sum(vals)
varianza_acumulada  = np.cumsum(varianza_individual)

# componente minima que alcanza 95%
k95 = np.where(varianza_acumulada >= 0.95)[0][0] + 1

fig = plt.figure(figsize=(14,8))

ax1 = plt.subplot(2,2,1)

ax1.plot(np.arange(1,n+1), vals, marker='o', linewidth=2)
ax1.set_title('Eigenvalores ')
ax1.set_xlabel('Componente principal')
ax1.set_ylabel('Eigenvalor')
ax1.set_xticks(np.arange(1,n+1))
ax1.grid(True, alpha=0.3)

ax2 = plt.subplot(2,2,2)

bars = ax2.bar(np.arange(1,n+1), 100*varianza_individual, alpha=0.8)

for i in range(n):
    ax2.text(i+1, 100*varianza_individual[i]+1,
             str(round(100*varianza_individual[i],1))+'%',
             ha='center', fontsize=9)

ax2.set_title('Varianza explicada por componente')
ax2.set_xlabel('Componente principal')
ax2.set_ylabel('%')
ax2.set_xticks(np.arange(1,n+1))
ax2.grid(True, axis='y', alpha=0.3)


ax3 = plt.subplot(2,1,2)

ax3.plot(np.arange(1,n+1), 100*varianza_acumulada,
         marker='o', linewidth=2)

ax3.axhline(95, linestyle='--', color='red')
ax3.axvline(k95, linestyle='--', color='green')

ax3.scatter(k95,100*varianza_acumulada[k95-1], s=80, zorder=3)

ax3.text(k95+0.15,
         100*varianza_acumulada[k95-1]-7,
         'k = '+str(k95),
         fontsize=10)

ax3.set_title('Varianza acumulada')
ax3.set_xlabel('Numero de componentes')
ax3.set_ylabel('% acumulado')
ax3.set_xticks(np.arange(1,n+1))
ax3.set_ylim(0,105)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Proyección de datos

Una vez obtenidos los eigenvectores, los datos se expresan en el nuevo sistema de coordenadas mediante la **proyección**:

$$
Z = X_c V
$$

donde:

- $X_c \in \mathbb{R}^{m \times n}$ es la matriz centrada,
- $V \in \mathbb{R}^{n \times n}$ tiene los eigenvectores como columnas,
- $Z \in \mathbb{R}^{m \times n}$ contiene las **coordenadas principales** de cada observación.

El elemento $z_{ij}$ indica cuánto participa la observación $i$ en la componente principal $j$.

Si se desea reducir la dimensión conservando solo las primeras $k$ componentes, se usa la submatriz:

$$
V_k = \begin{bmatrix} v_1 & v_2 & \cdots & v_k \end{bmatrix} \in \mathbb{R}^{n \times k}
$$

$$
Z_k = X_c V_k \in \mathbb{R}^{m \times k}
$$

Con $k < n$, cada observación queda representada con solo $k$ números en lugar de los $n$ originales.


In [ ]:
# Proyeccion de datos sobre las componentes principales
#
# Z = Xc * V   (producto matricial explícito con numpy.dot)
# Vk = primeras k columnas de V

# Proyeccion completa (todas las componentes)
Z_completo = np.dot(Xc, vecs)   # forma: (m, n) = (15, 4)

print('Dimension de Xc:', Xc.shape)
print('Dimension de V :', vecs.shape)
print('Dimension de Z :', Z_completo.shape)
print()
print('Primeras 5 filas de Z (coordenadas principales):')
print(np.round(Z_completo[:5, :], 4))
print()

# Proyeccion reducida con k = 1
k = 1
Vk = vecs[:, :k]              # columnas 0 hasta k-1; forma: (n, k) = (4, 1)
Zk = np.dot(Xc, Vk)           # forma: (m, k) = (15, 1)

print('Reduccion a k =', k, 'componente(s):')
print('  Dimension original de cada observacion: n =', n)
print('  Dimension reducida de cada observacion: k =', k)
print()
print('Primeras 5 coordenadas en la dimension reducida:')
print(np.round(Zk[:5, :], 4))
print()
print('Observacion: con k=1 conservamos', str(round(100 * varianza_acumulada[0], 2)) + '% de la varianza.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1,3, figsize=(18,5))

# usar solo primeras dos variables para visualizar
P = Xc[:, :2]

# vector PC1 en plano x1-x2
v = vecs[:2,0]
v = v / np.sqrt(np.sum(v**2))

# coordenadas proyectadas sobre PC1
scores = np.dot(P, v)

# puntos proyectados de regreso al plano
Proj = np.zeros((len(scores),2))
for i in range(len(scores)):
    Proj[i,:] = scores[i] * v

# ======================================
# 1. Datos originales
# ======================================
ax[0].scatter(P[:,0], P[:,1], s=65, alpha=0.85)

ax[0].set_title("1. Datos originales (2D)")
ax[0].set_xlabel("x1")
ax[0].set_ylabel("x2")
ax[0].grid(True, alpha=0.3)
ax[0].axis('equal')

# ======================================
# 2. PCA proyecta sobre la mejor linea
# ======================================
ax[1].scatter(P[:,0], P[:,1], s=45, alpha=0.35)

# linea principal
ax[1].plot([-5*v[0],5*v[0]], [-5*v[1],5*v[1]], linewidth=3)

# lineas de proyeccion
for i in range(len(P)):
    ax[1].plot([P[i,0], Proj[i,0]],
               [P[i,1], Proj[i,1]],
               alpha=0.35)

# puntos proyectados
ax[1].scatter(Proj[:,0], Proj[:,1], s=55, alpha=0.9)

ax[1].set_title("2. Proyeccion sobre PC1")
ax[1].set_xlabel("x1")
ax[1].set_ylabel("x2")
ax[1].grid(True, alpha=0.3)
ax[1].axis('equal')

# ======================================
# 3. Datos reducidos a una coordenada
# ======================================
ax[2].scatter(scores, np.zeros(len(scores)), s=70, alpha=0.9)

for i in range(len(scores)):
    ax[2].plot([scores[i], scores[i]], [0,0.03], alpha=0.4)

ax[2].set_title("3. Representacion 1D")
ax[2].set_xlabel("Coordenada PC1")
ax[2].set_yticks([])
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("PC1 conserva:",
      round(100*varianza_acumulada[0],2),
      "% de la varianza total")

#  Reconstrucción exacta

Cuando se utilizan **todas** las componentes principales, es posible recuperar exactamente la observación original.

Sea la proyección completa:

$$
z = V^T x
$$

La reconstrucción exacta se obtiene invirtiendo el cambio de base:

$$
x = Vz = VV^T x = Ix = x
$$

La igualdad $VV^T = I$ se debe a que los eigenvectores de una matriz simétrica forman una base **ortonormal**:

$$
v_i^T v_j = \begin{cases} 1 & \text{si } i = j \\ 0 & \text{si } i \neq j \end{cases}
$$

Esto significa que PCA **no destruye información** cuando se conservan todas las componentes; únicamente cambia la representación de los datos.


In [ ]:
# Demostracion de reconstruccion exacta
#
# Si usamos todos los eigenvectores, recuperamos Xc perfectamente.
# Reconstruccion: Xc_rec = Z * V^T

# Proyeccion completa
Z_completo = np.dot(Xc, vecs)          # (m, n)

# Reconstruccion: multiplicar por V^T
VT = vecs.T                            # transpuesta de V; forma: (n, n)
Xc_rec_exacta = np.dot(Z_completo, VT) # (m, n)

# Error de reconstruccion
error_max = np.max(np.abs(Xc - Xc_rec_exacta))
print('Error maximo de reconstruccion (usando todas las componentes):')
print('  max|Xc - Xc_rec| =', error_max)
print('  -> El error es numericamente cero: la reconstruccion es exacta.')
print()

# Verificar que VV^T = I
VVT = np.dot(vecs, vecs.T)
I_aprox = np.round(VVT, 10)
print('V * V^T (debe ser la identidad):')
print(np.round(I_aprox, 4))



#  Aproximación reducida

En la práctica, se usan solo las primeras $k < n$ componentes.

La reconstrucción aproximada de una observación centrada $x$ es:

$$
x_k = \sum_{i=1}^{k} z_i v_i = V_k z_k
$$

Para toda la matriz:

$$
X_k = Z_k V_k^T
$$

El **error de reconstrucción** para la observación $x$ es:

$$
\|x - x_k\|^2 = \sum_{i=k+1}^{n} z_i^2
$$

En promedio sobre todas las observaciones, el error cuadrático medio equivale a la varianza descartada:

$$
\overline{\|x - x_k\|^2} = \sum_{i=k+1}^{n} \lambda_i
$$

**PCA es óptima:** ninguna otra base de dimensión $k$ produce menor error cuadrático medio de reconstrucción. Este resultado se conoce como el **teorema de Eckart-Young**.


In [ ]:
# Reconstruccion aproximada y error segun numero de componentes
#
# Para cada k, proyectamos sobre las primeras k componentes y reconstruimos.
# Medimos el error cuadratico medio de reconstruccion.

errores_ecm = []
varianzas_exp = []

for k in range(1, n + 1):
    Vk  = vecs[:, :k]                       # primeras k columnas de V
    Zk  = np.dot(Xc, Vk)                    # proyeccion reducida: (m, k)
    VkT = Vk.T                              # (k, n)
    Xk  = np.dot(Zk, VkT)                  # reconstruccion aproximada: (m, n)

    diferencia = Xc - Xk                   # error por observacion y variable
    ecm = np.mean(diferencia ** 2)          # error cuadratico medio

    errores_ecm.append(ecm)
    varianzas_exp.append(100 * varianza_acumulada[k - 1])

print('k  | Varianza explicada | ECM de reconstruccion')
print('-' * 50)
for k in range(1, n + 1):
    print('k=' + str(k) + '  |  ' +
          str(round(varianzas_exp[k-1], 2)).rjust(8) + ' %       |  ' +
          str(round(errores_ecm[k-1], 6)))

# Visualizacion
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(range(1, n+1), varianzas_exp, marker='o', color='steelblue')
ax1.axhline(95, color='tomato', linestyle='--', label='95 %')
ax1.set_xlabel('k (numero de componentes)')
ax1.set_ylabel('Varianza acumulada (%)')
ax1.set_title('Varianza conservada')
ax1.set_xticks(range(1, n+1))
ax1.legend()

ax2.plot(range(1, n+1), errores_ecm, marker='o', color='tomato')
ax2.set_xlabel('k (numero de componentes)')
ax2.set_ylabel('Error cuadratico medio')
ax2.set_title('Error de reconstruccion')
ax2.set_xticks(range(1, n+1))

plt.tight_layout()
plt.show()

# Mostrar reconstruccion de la primera observacion para k=1 y k=2
print()
print('Primera observacion centrada (original):')
print('  ', np.round(Xc[0, :], 4))

for k in [1, 2]:
    Vk = vecs[:, :k]
    Zk = np.dot(Xc, Vk)
    Xk = np.dot(Zk, Vk.T)
    print('Reconstruccion con k=' + str(k) + ':')
    print('  ', np.round(Xk[0, :], 4))
    err = np.sqrt(np.sum((Xc[0, :] - Xk[0, :]) ** 2))
    print('  Error euclidiano:', round(err, 6))


# Relación entre PCA y SVD

La **Descomposición en Valores Singulares** (SVD) de la matriz centrada $X_c \in \mathbb{R}^{m \times n}$ está dada por:

$$
X_c = U \Sigma V^T
$$

donde:

- $U \in \mathbb{R}^{m \times m}$: columnas = vectores singulares izquierdos,
- $V \in \mathbb{R}^{n \times n}$: columnas = vectores singulares derechos,
- $\Sigma$: matriz diagonal con valores singulares $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$.

**Conexión con PCA:**

$$
X_c^T X_c = (U\Sigma V^T)^T (U\Sigma V^T) = V \Sigma^2 V^T
$$

Comparando con la descomposición espectral de la covarianza:

$$
S = \frac{1}{m-1} X_c^T X_c = V \left(\frac{\Sigma^2}{m-1}\right) V^T
$$

Por lo tanto:

$$
\lambda_i = \frac{\sigma_i^2}{m-1}
$$

Las columnas de $V$ son exactamente los eigenvectores de la matriz de covarianza. **PCA y SVD producen el mismo resultado.**

En la práctica, SVD se prefiere porque es numéricamente más estable que calcular explícitamente la covarianza y sus eigenvectores, especialmente cuando $n \gg m$.


In [ ]:
# Demostracion de la equivalencia PCA-SVD
#
# numpy.linalg.svd devuelve:
#   U     : vectores singulares izquierdos (columnas)
#   s     : valores singulares (como arreglo 1D, no matriz)
#   Vt    : V transpuesta (filas = vectores singulares derechos)
#
# Con full_matrices=False obtenemos la forma economica (compact SVD).

from numpy.linalg import svd

U, s, Vt = svd(Xc, full_matrices=False)

print('Dimensiones SVD:')
print('  U  :', U.shape)
print('  s  :', s.shape, '  (valores singulares)')
print('  Vt :', Vt.shape)
print()

# Los eigenvectores de PCA son las columnas de V = Vt^T
V_svd = Vt.T   # forma: (n, n)

# Los eigenvalores se obtienen de los valores singulares
lam_svd = (s ** 2) / (m - 1)

print('Comparacion eigenvalores:')
print('  i  | Eigen (via eig)  | Eigen (via SVD)')
print('  ' + '-' * 40)
for i in range(n):
    print('  ' + str(i+1) + '  |  ' +
          str(round(vals[i], 6)).rjust(14) + '  |  ' +
          str(round(lam_svd[i], 6)).rjust(14))

print()
print('Diferencia maxima entre eigenvalores:', round(np.max(np.abs(vals - lam_svd)), 10))
print('-> Los resultados son numericamente identicos.')
print()

# Los eigenvectores pueden diferir en signo (ambas versiones son validas)
# Comparar columna a columna (en valor absoluto)
print('Comparacion de eigenvectores (valor absoluto, primera columna):')
print('  via eig:')
print('  ', np.round(np.abs(vecs[:, 0]), 6))
print('  via SVD:')
print('  ', np.round(np.abs(V_svd[:, 0]), 6))


#  Aplicación a imágenes faciales: Eigenfaces

Las imágenes digitales son uno de los ejemplos más intuitivos de datos de alta dimensión.

Una imagen en escala de grises de $64 \times 64$ píxeles se puede tratar como un vector de:

$$
n = 64 \times 64 = 4096 \text{ variables}
$$

Una colección de $m$ imágenes forma entonces la matriz:

$$
X \in \mathbb{R}^{m \times 4096}
$$

Aquí, $n \gg m$ en la mayoría de los casos prácticos.

**Eigenfaces** (Turk y Pentland, 1991) es la aplicación directa de PCA a imágenes de rostros:

1. Se centra la colección de imágenes (se resta el **rostro promedio**).
2. Se calcula la SVD de la matriz centrada.
3. Los primeros vectores singulares derechos son las **eigenfaces**: direcciones de máxima variabilidad entre rostros.
4. Cada rostro puede reconstruirse como combinación lineal de las eigenfaces.

La clave es que con pocas eigenfaces (decenas) es posible representar aceptablemente una colección de cientos de rostros.

**Referencia:**  
Turk, M., & Pentland, A. (1991). Eigenfaces for recognition. *Journal of Cognitive Neuroscience, 3*(1), 71–86. https://doi.org/10.1162/jocn.1991.3.1.71


In [ ]:
# Eigenfaces: Paso 1 — Cargar y visualizar el conjunto de datos
#
# Usamos el dataset Olivetti Faces (sklearn).
# Contiene 400 imagenes de 40 personas (10 imagenes por persona).
# Cada imagen es de 64x64 pixeles en escala de grises.
# Los valores de los pixeles estan normalizados entre 0 y 1.

from sklearn.datasets import fetch_olivetti_faces

datos = fetch_olivetti_faces(shuffle=True, random_state=0)

# X contiene las imagenes "aplanadas": cada fila es una imagen de 4096 pixeles
X_faces = datos.data    # forma: (400, 4096)
# images contiene las imagenes con su forma original para visualizar
imagenes = datos.images # forma: (400, 64, 64)

m_faces = X_faces.shape[0]  # 400 imagenes
n_faces = X_faces.shape[1]  # 4096 pixeles por imagen

print('Dimensiones de la matriz de datos:')
print('  Filas (imagenes):  ', m_faces)
print('  Columnas (pixeles):', n_faces)
print()
print('Cada imagen tiene', int(n_faces ** 0.5), 'x', int(n_faces ** 0.5), 'pixeles.')
print('Rango de valores de pixel: [', round(X_faces.min(), 3), ',', round(X_faces.max(), 3), ']')

# Visualizar las primeras 10 imagenes
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, img in zip(axes.ravel(), imagenes[:10]):
    ax.imshow(img, cmap='gray')
    ax.axis('off')
plt.suptitle('Muestra de 10 rostros del dataset Olivetti', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Eigenfaces: Paso 2 — Centrado de los datos
#
# Calculamos el rostro promedio y lo restamos a cada imagen.
# El resultado es la variabilidad de cada rostro respecto al promedio.

# Media de cada pixel a lo largo de todas las imagenes
mu_faces = np.mean(X_faces, axis=0)   # forma: (4096,)

# Centrar: restar la media a cada fila
Xc_faces = X_faces - mu_faces         # forma: (400, 4096)

# Visualizar el rostro promedio
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))

ax1.imshow(mu_faces.reshape(64, 64), cmap='gray')
ax1.set_title('Rostro promedio')
ax1.axis('off')

# Un ejemplo de dato centrado (variacion respecto al promedio)
ax2.imshow(Xc_faces[0].reshape(64, 64), cmap='RdGy')
ax2.set_title('Primer rostro centrado (diferencia con el promedio)')
ax2.axis('off')

plt.tight_layout()
plt.show()

print('Verificacion: media de los datos centrados (debe ser ~0):')
print('  Media global de Xc_faces:', round(np.mean(Xc_faces), 10))


In [ ]:
# Eigenfaces: Paso 3 — SVD de la matriz centrada
#
# Calculamos la SVD de Xc_faces para obtener las componentes principales.
# Usamos full_matrices=False para obtener la forma compacta (economica).
#
# Xc_faces = U * diag(s) * Vt
#
#   U  : (400, 400) — coordenadas de las imagenes en el espacio principal
#   s  : (400,)     — valores singulares (importancia de cada componente)
#   Vt : (400, 4096)— eigenfaces (filas = componentes principales)
#
# NOTA: como m=400 < n=4096, la SVD compacta devuelve
#       min(m,n) = 400 componentes, no 4096.

from numpy.linalg import svd

print('Calculando SVD ...')
U_f, s_f, Vt_f = svd(Xc_faces, full_matrices=False)

print('Hecho.')
print()
print('Dimensiones de la SVD:')
print('  U_f  :', U_f.shape)
print('  s_f  :', s_f.shape, '  (valores singulares)')
print('  Vt_f :', Vt_f.shape)
print()

# Los eigenvalores de la covarianza son sigma_i^2 / (m-1)
m_faces = Xc_faces.shape[0]
lam_faces = (s_f ** 2) / (m_faces - 1)

# Varianza explicada
var_exp_faces = lam_faces / np.sum(lam_faces)
var_acum_faces = np.cumsum(var_exp_faces)

print('Valores singulares (primeros 10):')
print(np.round(s_f[:10], 2))
print()
print('Varianza acumulada:')
for k in [1, 5, 10, 20, 50, 100, 200]:
    print('  Primeras ' + str(k).rjust(3) + ' componentes: ' +
          str(round(100 * var_acum_faces[k-1], 2)) + ' %')


In [ ]:
# Eigenfaces: Paso 4 — Visualizar las eigenfaces
#
# Las eigenfaces son las filas de Vt_f.
# Cada eigenface es un vector de 4096 numeros que, al reorganizarse
# en una matriz de 64x64, forma una imagen que representa
# una 'direccion de variacion' entre los rostros del conjunto.
#
# Las primeras eigenfaces capturan las diferencias mas grandes
# entre los rostros (iluminacion, orientacion, etc.).

k_mostrar = 10   # cuantas eigenfaces visualizar

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.ravel()):
    # La eigenface i es la fila i de Vt_f, reorganizada en 64x64
    eigenface_i = Vt_f[i, :].reshape(64, 64)
    ax.imshow(eigenface_i, cmap='RdGy')
    ax.set_title('Eigenface ' + str(i+1) +
                 '\n(' + str(round(100 * var_exp_faces[i], 1)) + '% var.)',
                 fontsize=8)
    ax.axis('off')

plt.suptitle('Las primeras 10 eigenfaces (componentes principales de los rostros)',
             fontsize=11)
plt.tight_layout()
plt.show()

print('  Eigenface 1: captura la mayor variacion global (iluminacion, tono).')
print('  Eigenfaces siguientes: capturan variaciones mas sutiles.')
print('  Cualquier rostro se puede aproximar como:')
print('    rostro = rostro_promedio + z1*eigenface1 + z2*eigenface2 + ...')


In [ ]:
# Eigenfaces: Paso 5 — Reconstruccion de un rostro
#
# Mostramos como se reconstruye un rostro paso a paso.
#
# El rostro centrado x_c tiene 4096 dimensiones.
# La proyeccion sobre las primeras k eigenfaces es:
#
#   z_k = Xc * Vk^T   (coordenadas en el espacio reducido)
#
# donde Vk tiene las primeras k filas de Vt_f y
# Vk^T tiene las primeras k columnas de V = Vt_f^T.
#
# La reconstruccion aproximada del dato centrado es:
#
#   Xc_rec = z_k * Vk  = Xc * Vk^T * Vk
#
# Para recuperar la imagen original se suma el rostro promedio:
#
#   x_rec = Xc_rec + mu_faces

idx_rostro = 5   # cual imagen reconstruir (0 a 399)
ks_probar = [1, 5, 10, 20, 50, 100]   # cuantas eigenfaces usar

fig, axes = plt.subplots(1, len(ks_probar) + 1, figsize=(14, 3))

# Imagen original
axes[0].imshow(X_faces[idx_rostro].reshape(64, 64), cmap='gray')
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

for j, k in enumerate(ks_probar):
    # Seleccionar las primeras k eigenfaces (filas de Vt_f)
    Vk   = Vt_f[:k, :]         # forma: (k, 4096)
    VkT  = Vk.T                 # forma: (4096, k)

    # Proyeccion: coordenadas del rostro centrado en el espacio de k dimensiones
    # Usamos solo la fila correspondiente al rostro elegido
    xc_i = Xc_faces[idx_rostro, :]       # forma: (4096,)
    z_k  = np.dot(xc_i, VkT)            # forma: (k,)

    # Reconstruccion del dato centrado
    xc_rec = np.dot(z_k, Vk)            # forma: (4096,)

    # Recuperar la imagen original sumando el promedio
    x_rec = xc_rec + mu_faces            # forma: (4096,)

    # Error de reconstruccion
    error = np.sqrt(np.sum((X_faces[idx_rostro] - x_rec) ** 2))

    axes[j + 1].imshow(x_rec.reshape(64, 64), cmap='gray', vmin=0, vmax=1)
    axes[j + 1].set_title('k=' + str(k) + '\nerror=' + str(round(error, 3)), fontsize=8)
    axes[j + 1].axis('off')

plt.suptitle('Reconstruccion del rostro ' + str(idx_rostro) +
             ' con distintas cantidades de eigenfaces', fontsize=10)
plt.tight_layout()
plt.show()

print()
print('Interpretacion:')
print('  Con muy pocas eigenfaces el rostro es irreconocible.')
print('  Con 10-20 eigenfaces la identidad ya es reconocible.')
print('  Con 100+ eigenfaces la calidad se acerca mucho al original.')
print()
print('Relacion almacenamiento:')
print('  Original: 4096 numeros por imagen.')
print('  Con k=20: 20 coordenadas + compartir 20 eigenfaces de 4096 pixeles.')
print('  Ahorro por imagen: de 4096 a 20 numeros (ratio 1 : ' + str(n_faces // 20) + ').')


#  Ejercicio 

Determinar experimentalmente cuántas eigenfaces son suficientes para representar los rostros y evaluar la calidad de la reconstrucción.


## Parte A — Varianza explicada y selección de $k$

**Objetivo:** encontrar el menor $k$ tal que la varianza acumulada sea al menos del $90\%$.

La varianza explicada por las primeras $k$ componentes ya está calculada en el arreglo `var_acum_faces`.

**Instrucciones:**

1. Recorra los valores de `var_acum_faces` y encuentre el primer índice donde supere $0.90$.
2. Imprima ese valor de $k$ y el porcentaje exacto alcanzado.
3. Grafique la curva de varianza acumulada marcando ese $k$ con una línea vertical.


In [ ]:
# Parte A — Encuentre el k minimo para 90% de varianza
#
# var_acum_faces ya esta calculado arriba.
# Tip: use un ciclo for o numpy.argmax para encontrar el primer indice
# donde var_acum_faces supere 0.90.

umbral = 0.90  # 90 % de varianza

# --- Variar k ---
k_minimo = 1

print('Menor k con', int(umbral * 100), '% de varianza:')
print('  k =', k_minimo)
print('  Varianza acumulada exacta:', round(100 * var_acum_faces[k_minimo - 1], 2), '%')

# --- Grafica ---
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(var_acum_faces) + 1), 100 * var_acum_faces, marker ="*", color='steelblue', lw=1.5)
plt.axhline(umbral * 100, color='tomato', linestyle='--', label=str(int(umbral*100)) + '% umbral')
plt.axvline(k_minimo, color='green', linestyle='--', label='k = ' + str(k_minimo))
plt.xlabel('Numero de componentes k')
plt.ylabel('Varianza acumulada (%)')
plt.title('Varianza acumulada en Olivetti Faces')
plt.legend()
plt.xlim([1, 200])
plt.tight_layout()
plt.show()


## Parte B — Reconstrucción de cinco rostros distintos

**Objetivo:** reconstruir cinco rostros diferentes usando el $k$ encontrado en la Parte A y evaluar visualmente la calidad.

**Instrucciones:**

1. Elija cinco índices de imágenes distintos.
2. Para cada imagen, realice la reconstrucción usando exactamente $k_{\min}$ eigenfaces.
3. Muestre la imagen original y la reconstruida lado a lado.
4. Calcule el error euclidiano $\|x - x_{\text{rec}}\|$ para cada rostro.


In [ ]:
# Parte B — Reconstruccion de cinco rostros con k_minimo eigenfaces

indices_rostros = [ ]   # cinco rostros distintos
k_usar = k_minimo                           # k encontrado en la Parte A

# Preparar las k primeras eigenfaces
Vk   = Vt_f[:k_usar, :]   # (k, 4096)
VkT  = Vk.T               # (4096, k)

fig, axes = plt.subplots(len(indices_rostros), 2, figsize=(6, 3 * len(indices_rostros)))

for fila, idx in enumerate(indices_rostros):

    # Dato centrado del rostro idx
    xc_i = Xc_faces[idx, :]           # (4096,)

    # Proyeccion sobre las primeras k eigenfaces
    z_k   = np.dot(xc_i, VkT)         # (k,)

    # Reconstruccion centrada
    xc_rec = np.dot(z_k, Vk)          # (4096,)

    # Sumar promedio para recuperar la imagen
    x_rec = xc_rec + mu_faces          # (4096,)

    # Error ... tu codigo calcular e imprimir error
   

    # Mostrar
    axes[fila, 0].imshow(X_faces[idx].reshape(64, 64), cmap='gray', vmin=0, vmax=1)
    axes[fila, 0].set_title('Original (indice ' + str(idx) + ')', fontsize=8)
    axes[fila, 0].axis('off')

    axes[fila, 1].imshow(x_rec.reshape(64, 64), cmap='gray', vmin=0, vmax=1)
    axes[fila, 1].set_title('Reconstruido k=' + str(k_usar) +
                            '  error=' + str(round(error, 3)), fontsize=8)
    axes[fila, 1].axis('off')

plt.suptitle('Reconstruccion de 5 rostros con k=' + str(k_usar) + ' eigenfaces',
             fontsize=10)
plt.tight_layout()
plt.show()


## Parte C — Curva de error vs. compresión (conexión con Diseño Experimental)

**Objetivo:** evaluar sistemáticamente el compromiso entre número de componentes y calidad de reconstrucción.

Esto es un ejemplo directo de **diseño experimental de un factor**: el factor es $k$ (número de eigenfaces) y la respuesta es el error cuadrático medio de reconstrucción.

**Instrucciones:**

1. Defina una grilla de valores de $k$: `ks = [ , , ,]`.
2. Para cada $k$, calcule el error cuadrático medio de reconstrucción **sobre los 400 rostros**.
3. Grafique error vs. $k$ y varianza explicada vs. $k$ en la misma figura.
4. Responda: ¿A partir de qué $k$ el error deja de mejorar notablemente? ¿Coincide con la varianza acumulada?


In [ ]:
# Parte C — Curva de error cuadratico medio vs. numero de eigenfaces

ks_grid = []

ecm_lista = []

for k in ks_grid:
    Vk   = Vt_f[:k, :]        # primeras k eigenfaces: (k, 4096)
    VkT  = Vk.T               # (4096, k)

    # Proyeccion de TODOS los rostros
    Z_k  = np.dot(Xc_faces, VkT)    # (400, k)

    # Reconstruccion de todos los rostros
    Xc_rec = np.dot(Z_k, Vk)        # (400, 4096)

    # Calcula el error cuadratico medio sobre todos los pixeles y todas las imagenes


    print('k =', str(k).rjust(3), '  ECM =', str(round(ecm, 6)).rjust(10),
          '  Var. exp. =', str(round(100 * var_acum_faces[k-1], 1)).rjust(6), '%')

# Visualizacion
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ks_grid, ecm_lista, marker='o', color='tomato')
ax1.set_xlabel('k (numero de eigenfaces)')
ax1.set_ylabel('Error cuadratico medio')
ax1.set_title('Error de reconstruccion vs. k')

ax2.plot(ks_grid, [100 * var_acum_faces[k-1] for k in ks_grid], marker='o', color='steelblue')
ax2.axhline(90, linestyle='--', color='gray', label='90 %')
ax2.set_xlabel('k (numero de eigenfaces)')
ax2.set_ylabel('Varianza acumulada (%)')
ax2.set_title('Varianza explicada vs. k')
ax2.legend()

plt.tight_layout()
plt.show()

print()

print('  El error decrece rapidamente hasta cierto k y luego se estabiliza.')
print('  Ese "codo" en la curva indica el punto de rendimientos decrecientes:')
print('  agregar mas eigenfaces aporta cada vez menos mejora.')
print('  Esta idea es fundamental en optimizacion de modelos.')


# Conclusiones

A lo largo de este notebook se desarrollaron los siguientes conceptos:

**Fundamentos matemáticos:**

- La matriz de datos $X \in \mathbb{R}^{m \times n}$ organiza $m$ observaciones con $n$ variables cada una.
- El centrado $X_c = X - \mathbf{1}\mu^T$ elimina la media y coloca el análisis en términos de variabilidad.
- La matriz de covarianza $S = X_c^T X_c / (m-1)$ cuantifica las dependencias lineales entre variables.
- Los eigenvectores de $S$ definen las **componentes principales**: direcciones de máxima varianza.
- Los eigenvalores $\lambda_i$ miden la varianza en cada dirección principal.

**Algoritmo PCA:**

1. Centrar los datos.
2. Calcular la covarianza o aplicar SVD directamente.
3. Ordenar componentes por varianza descendente.
4. Proyectar: $Z_k = X_c V_k$.
5. Reconstruir: $\hat{X}_k = Z_k V_k^T + \mu$.

**Relación PCA-SVD:**

- $\lambda_i = \sigma_i^2 / (m-1)$. Las columnas de $V$ son los eigenvectores de $S$.
- SVD es el método preferido por su estabilidad numérica.

**Aplicación Eigenfaces:**

- Cada imagen es un punto en un espacio de 4096 dimensiones.
- Las eigenfaces son las direcciones de mayor variación entre rostros.
- Con pocas decenas de eigenfaces se puede representar aceptablemente un conjunto de cientos de imágenes.

**Conexión con Diseño Experimental:**

- La selección del número de componentes $k$ es un problema de optimización con un factor continuo y una respuesta cuantificable (error o varianza explicada).
- La curva de error vs. $k$ muestra el principio de rendimientos decrecientes: incrementar $k$ más allá del "codo" no mejora significativamente la representación.

**Referencias:**

Turk, M., & Pentland, A. (1991). Eigenfaces for recognition. *Journal of Cognitive Neuroscience, 3*(1), 71–86. https://doi.org/10.1162/jocn.1991.3.1.71

Jolliffe, I. T., & Cadima, J. (2016). Principal component analysis: a review and recent developments. *Philosophical Transactions of the Royal Society A, 374*(2065), 20150202. https://doi.org/10.1098/rsta.2015.0202

Wall, M. E., Rechtsteiner, A., & Rocha, L. M. (2003). Singular value decomposition and principal component analysis. In D. P. Berrar, W. Dubitzky, & M. Granzow (Eds.), *A practical approach to microarray data analysis* (pp. 91–109). Springer. https://doi.org/10.1007/0-306-47815-3_5
